In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path(r"C:\Users\Robin Yadav\oulad-early-warning")
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
CUTOFF = 28
key = ["code_module", "code_presentation", "id_student"]

base = pd.read_csv(PROCESSED / f"base_day{CUTOFF}.csv")
print("base:", base.shape)

base: (27538, 15)


In [2]:
clicks = pd.read_csv(
    RAW / "studentVle.csv",
    dtype={"code_module": "category", "code_presentation": "category",
           "id_student": "int32", "id_site": "int32",
           "date": "int16", "sum_click": "int32"},
)
c = clicks[clicks["date"] <= CUTOFF].copy()
print("max click day used:", c["date"].max())  # must be 28

c["period"] = np.select([c["date"] < 0, c["date"] <= 13], ["pre", "early"], "late")

g = c.groupby(key, observed=True).agg(
    total_clicks=("sum_click", "sum"),
    active_days=("date", "nunique"),
    n_sites=("id_site", "nunique"),
    last_active_day=("date", "max"),
).reset_index()

p = (c.groupby(key + ["period"], observed=True)["sum_click"].sum()
       .unstack(fill_value=0).reset_index())
p = p.rename(columns={"pre": "clicks_pre", "early": "clicks_days0_13", "late": "clicks_days14_28"})

click_feats = g.merge(p, on=key, how="left")
for col in ["code_module", "code_presentation"]:
    click_feats[col] = click_feats[col].astype(str)
click_feats["click_trend"] = click_feats["clicks_days14_28"] - click_feats["clicks_days0_13"]
click_feats["days_since_last_active"] = CUTOFF - click_feats["last_active_day"]
click_feats = click_feats.drop(columns="last_active_day")
print(click_feats.shape)
click_feats.head()

max click day used: 28
(28792, 11)


,code_module,code_presentation,id_student,total_clicks,active_days,n_sites,clicks_days0_13,clicks_days14_28,clicks_pre,click_trend,days_since_last_active
0,AAA,2013J,11391,401,8,24,203,100,98,-103,10
1,AAA,2013J,28400,618,19,34,241,162,215,-79,0
2,AAA,2013J,30268,281,12,22,179,0,102,-179,16
3,AAA,2013J,31604,494,22,31,176,149,169,-27,0
4,AAA,2013J,32885,567,24,34,177,95,295,-82,2


In [3]:
assess = pd.read_csv(RAW / "assessments.csv")
sa = pd.read_csv(RAW / "studentAssessment.csv")

due = assess[assess["date"].notna() & (assess["date"] <= CUTOFF)]
print("assessments due by cutoff:", len(due))

sub = sa.merge(due[["id_assessment", "code_module", "code_presentation"]], on="id_assessment")
sub = sub[sub["date_submitted"] <= CUTOFF]

a = sub.groupby(key).agg(
    n_submitted=("id_assessment", "nunique"),
    mean_score=("score", "mean"),
).reset_index()

n_due = due.groupby(["code_module", "code_presentation"]).size().rename("n_assessments_due").reset_index()
print(n_due)

assessments due by cutoff: 17
   code_module code_presentation  n_assessments_due
0          AAA             2013J                  1
1          AAA             2014J                  1
2          BBB             2013B                  1
3          BBB             2013J                  1
4          BBB             2014B                  1
5          BBB             2014J                  1
6          CCC             2014B                  1
7          CCC             2014J                  1
8          DDD             2013B                  2
9          DDD             2013J                  1
10         DDD             2014B                  1
11         DDD             2014J                  1
12         FFF             2013B                  1
13         FFF             2013J                  1
14         FFF             2014B                  1
15         FFF             2014J                  1


In [4]:
feats = (base.drop(columns=["date_unregistration", "final_result"])
             .merge(click_feats, on=key, how="left")
             .merge(a, on=key, how="left")
             .merge(n_due, on=["code_module", "code_presentation"], how="left"))

zero_cols = ["total_clicks", "active_days", "n_sites", "clicks_pre", "clicks_days0_13",
             "clicks_days14_28", "click_trend", "n_submitted", "n_assessments_due"]
feats[zero_cols] = feats[zero_cols].fillna(0)
feats["days_since_last_active"] = feats["days_since_last_active"].fillna(CUTOFF + 1)
feats["no_clicks_flag"] = (feats["total_clicks"] == 0).astype(int)

print(feats.shape)
print(feats.isna().sum()[lambda s: s > 0])
print("leak check:", [x for x in ["final_result", "date_unregistration"] if x in feats.columns])

feats.to_csv(PROCESSED / f"features_day{CUTOFF}.csv", index=False)

(27538, 25)
imd_band             1028
date_registration       7
mean_score           7464
dtype: int64
leak check: []
